In [ ]:
# RQ1: Baseline Performance
# How effectively can baseline supervised learning models (Linear Regression, Decision Tree, k-NN)
# solve the prediction problem on the Marketing and Product Performance Dataset?

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load dataset (update path if needed)
df = pd.read_csv('/kaggle/input/marketing-and-product-performance-dataset/marketing_and_product_performance.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
# Preprocessing
for col in ['Subscription_Tier', 'Common_Keywords']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

drop_cols = ['Campaign_ID', 'Product_ID', 'Customer_ID', 'Flash_Sale_ID', 'Bundle_ID']
df = df.drop(columns=drop_cols)

X = df.drop(columns=['Units_Sold'])
y = df['Units_Sold']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
# Train and evaluate baseline models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'K-NN (k=5)': KNeighborsRegressor(n_neighbors=5)
}

results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    cv = cross_val_score(model, X_train_s, y_train, cv=5, scoring='r2').mean()
    results.append({'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4), 'CV_R2': round(cv,4)})

res_df = pd.DataFrame(results)
print(res_df)
res_df.to_csv('RQ1_baseline_results.csv', index=False)

In [ ]:
# Figure: Actual vs Predicted scatter plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800']
for i, (name, model) in enumerate(models.items()):
    preds = model.predict(X_test_s)
    axes[i].scatter(y_test, preds, alpha=0.3, s=10, color=colors[i])
    mn, mx = min(y_test.min(), preds.min()), max(y_test.max(), preds.max())
    axes[i].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect fit')
    axes[i].set_xlabel('Actual Units Sold')
    axes[i].set_ylabel('Predicted Units Sold')
    r2 = r2_score(y_test, preds)
    axes[i].set_title(f'{name}\nR²={r2:.4f}')
    axes[i].legend()
plt.suptitle('RQ1: Baseline Model Performance — Actual vs Predicted', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('RQ1_baseline_performance.pdf', dpi=150, bbox_inches='tight')
plt.show()